# Exploratory Data Analysis (EDA) for Project Datasets
This notebook loads the four datasets defined in `conf/base/catalog.yml` using Kedro's global catalog, performs basic inspection, and creates visualizations.

In [ ]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Load Kedro catalog
from kedro.framework.session import KedroSession
session = KedroSession.create()
context = session.load_context()
catalog = context.catalog


In [ ]:
# Load datasets
employee_df = catalog.load("employee_data")
engagement_df = catalog.load("employee_engagement_survey_data")
recruitment_df = catalog.load("recruitment_data")
training_df = catalog.load("training_and_development_data")


In [ ]:
# Basic inspection for each dataframe
def inspect(df, name):
    print(f'--- {name} ---')
    display(df.info())
    display(df.describe())
    print('Null values per column:')
    print(df.isnull().sum())
    print('Duplicate rows:', df.duplicated().sum())
    print('\n')

inspect(employee_df, 'employee_data')
inspect(engagement_df, 'employee_engagement_survey_data')
inspect(recruitment_df, 'recruitment_data')
inspect(training_df, 'training_and_development_data')


## Visualizations
Below are three example visualizations. Adjust column names as appropriate for your datasets.

In [ ]:
# 1. Salary distribution (if 'salary' column exists)
if 'salary' in employee_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(employee_df['salary'].dropna(), kde=True, bins=30)
    plt.title('Salary Distribution')
    plt.xlabel('Salary')
    plt.show()
else:
    print('Column "salary" not found in employee_data')


In [ ]:
# 2. Performance by Department (example)
if {'department', 'performance_score'}.issubset(employee_df.columns):
    plt.figure(figsize=(10,6))
    perf_by_dept = employee_df.groupby('department')['performance_score'].mean().reset_index()
    sns.barplot(data=perf_by_dept, x='department', y='performance_score')
    plt.title('Average Performance by Department')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('Required columns for performance plot not found')


In [ ]:
# 3. Recruitment sources pie chart (example)
if 'source' in recruitment_df.columns:
    plt.figure(figsize=(6,6))
    recruitment_counts = recruitment_df['source'].value_counts()
    recruitment_counts.plot.pie(autopct='%1.1f%%', startangle=90)
    plt.title('Recruitment Sources Distribution')
    plt.ylabel('')
    plt.show()
else:
    print('Column "source" not found in recruitment_data')


## 📊 Análisis Estratégico para Recursos Humanos (HR)

A continuación, presentaremos las historias visuales y conclusiones solicitadas en la evaluación, cargando primero nuestra tabla maestra integrada (`master_table`) que contiene todas las variables cruzadas y calculadas en los pipelines anteriores.

In [ ]:
# Cargamos la master table generada por nuestro pipeline de transformación
master_table = catalog.load('master_table')

# Verificamos rápidamente las columnas de interés
display(master_table[['Performance Score', 'Pay Zone', 'Seniority_Years', 'Total Training Cost']].head())

### 1. Brecha Salarial y Desempeño
**Gráfico**: Boxplot de Relación entre `Performance Score` contra `Pay Zone` (Bandas Salariales).

**Pregunta a responder**: ¿Existe una brecha salarial o relación directa entre el nivel de desempeño y la banda salarial en la que se encuentran los empleados?

In [ ]:
plt.figure(figsize=(10, 6))
# Usamos un boxplot para ver la distribución de Performance Score dentro de cada Pay Zone
sns.boxplot(data=master_table, x='Pay Zone', y='Performance Score', palette='Set2')
plt.title('Brecha Salarial: Distribución de Desempeño por Banda Salarial', fontsize=14)
plt.xlabel('Banda Salarial (Pay Zone)', fontsize=12)
plt.ylabel('Puntaje de Desempeño (Performance Score)', fontsize=12)
plt.tight_layout()
plt.show()

**💡 Conclusión Analítica (HR):**
Este Boxplot nos permite observar la distribución de los puntajes de desempeño a través de las diferentes bandas salariales. 
Si observamos que las bandas salariales más altas no tienen necesariamente los puntajes de desempeño más altos (o si la mediana es similar en todas las zonas), esto podría indicar que la estructura salarial no está fuertemente alineada al mérito o rendimiento, sino que podría estar más ligada a la antigüedad o la posición jerárquica inicial. Por el contrario, si las zonas más altas concentran consistentemente los mejores desempeños, la política de compensaciones refleja un esquema saludable de meritocracia.

### 2. Distribución de Antigüedad
**Gráfico**: Histograma de Distribución de Antigüedad (`Seniority_Years`).

**Pregunta a responder**: ¿Cómo se distribuye el tiempo de servicio de nuestra fuerza laboral? ¿Tenemos una empresa joven o de larga trayectoria?

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(master_table['Seniority_Years'].dropna(), bins=20, kde=True, color='teal')
plt.title('Distribución de Antigüedad de los Empleados', fontsize=14)
plt.xlabel('Años de Antigüedad (Seniority_Years)', fontsize=12)
plt.ylabel('Cantidad de Empleados', fontsize=12)
plt.axvline(master_table['Seniority_Years'].mean(), color='red', linestyle='dashed', linewidth=2, label='Promedio')
plt.legend()
plt.tight_layout()
plt.show()

**💡 Conclusión Analítica (HR):**
El histograma nos revela la forma en que el talento se concentra por años de antigüedad. 
Una distribución sesgada hacia la izquierda indicaría una empresa con altas tasas de contratación reciente o alta rotación (la mayoría del personal tiene pocos años). Una distribución normal u homogénea muestra retención de talento a largo plazo. La línea roja nos ayuda a identificar rápidamente el promedio de pertenencia, un KPI vital para proyectar riesgos de fuga de talento o planificar planes de carrera.

### 3. Impacto de la Capacitación en el Rendimiento
**Gráfico**: Gráfico de Dispersión / Impacto (`Total Training Cost` vs `Performance Score`).

**Pregunta a responder**: ¿Los empleados con mayor presupuesto de capacitación asignado rinden mejor?

In [ ]:
plt.figure(figsize=(10, 6))
# Gráfico de dispersión con línea de tendencia (regplot)
sns.regplot(data=master_table, x='Total Training Cost', y='Performance Score', 
            scatter_kws={'alpha':0.5, 'color':'indigo'}, line_kws={'color':'red'})
plt.title('Impacto de la Capacitación en el Rendimiento', fontsize=14)
plt.xlabel('Costo Total de Capacitación (USD)', fontsize=12)
plt.ylabel('Puntaje de Desempeño (Performance Score)', fontsize=12)
plt.tight_layout()
plt.show()

**💡 Conclusión Analítica (HR):**
La gráfica de dispersión y su línea de tendencia (roja) nos responden directamente la rentabilidad del presupuesto de capacitación en el desempeño individual. 
- Si la línea de tendencia es **positiva y pronunciada**, confirmamos que la inversión en capacitación impacta directamente y eleva el desempeño de los colaboradores (ROI positivo).
- Si la línea es **plana o no hay un patrón claro**, significa que destinar más dinero a capacitar a un empleado no está garantizando un mejor performance, lo que obliga a HR a revisar la calidad, pertinencia o los proveedores de los cursos que se están impartiendo.